# 71 Job · Tarea 02 · Control de Calidad

Esta tarea valida la capa Bronze y alimenta la decisión `if/else` del Lakeflow Job. Vas a convertir reglas de calidad en columnas booleanas, separar datos válidos y rechazos, y publicar métricas como *task values*. Cubre los objetivos DCEA de **data quality checks** y **control flows con retries y condiciones**.

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [ ]:
TABLA_BRONZE = f"{CATALOG}.{SCHEMA}.movielens_bronze"
assert spark.catalog.tableExists(TABLA_BRONZE), (
    f"No existe {TABLA_BRONZE}. Ejecutá primero la tarea 01_ingesta_bronze "
    "o el notebook 70_Job_01_Ingesta_Bronze."
)

try:
    filas_ingeridas = dbutils.jobs.taskValues.get(
        taskKey="01_ingesta_bronze", key="filas_ingeridas", default=-1)
except Exception:
    filas_ingeridas = -1
print(f"Task value recibido: filas_ingeridas={filas_ingeridas}")

In [ ]:
dbutils.widgets.text("inyectar_error", "false", "Inyectar error de calidad")
inyectar_error = dbutils.widgets.get("inyectar_error").strip().lower() == "true"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table(TABLA_BRONZE)
if inyectar_error:
    # Una fila con rating inválido fuerza de manera reproducible la rama false del gate.
    corrupta = bronze.limit(1).withColumn("rating", F.lit(99.0))
    bronze = bronze.unionByName(corrupta)
    print("Se inyectó una fila corrupta para la demostración.")

In [ ]:
ventana_duplicados = Window.partitionBy("userId", "movieId").orderBy(
    F.col("timestamp").desc_nulls_last(), F.col("_ingest_ts").desc_nulls_last())

evaluadas = (bronze
    .withColumn("_numero_duplicado", F.row_number().over(ventana_duplicados))
    .withColumn("error_rating_fuera_rango", F.col("rating").isNotNull() & ~F.col("rating").between(0.5, 5.0))
    .withColumn("error_rating_nulo", F.col("rating").isNull())
    .withColumn("error_id_nulo", F.col("userId").isNull() | F.col("movieId").isNull())
    .withColumn("error_duplicado", F.col("_numero_duplicado") > 1)
    .withColumn("motivo_rechazo", F.concat_ws("; ",
        F.when(F.col("error_rating_fuera_rango"), F.lit("rating fuera de [0.5, 5.0]")),
        F.when(F.col("error_rating_nulo"), F.lit("rating nulo")),
        F.when(F.col("error_id_nulo"), F.lit("userId o movieId nulo")),
        F.when(F.col("error_duplicado"), F.lit("duplicado de userId/movieId"))))
    .withColumn("es_invalida", F.length("motivo_rechazo") > 0))

columnas_tecnicas = ["_numero_duplicado", "error_rating_fuera_rango", "error_rating_nulo",
                     "error_id_nulo", "error_duplicado", "es_invalida"]
silver = evaluadas.filter(~F.col("es_invalida")).drop("motivo_rechazo", *columnas_tecnicas)
rechazos = evaluadas.filter(F.col("es_invalida")).drop(*columnas_tecnicas)

(silver.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.movielens_silver"))
(rechazos.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.movielens_rechazos"))

In [ ]:
filas_evaluadas = evaluadas.count()
filas_malas = rechazos.count()
tasa_rechazo = round(filas_malas / filas_evaluadas, 4) if filas_evaluadas else 0.0

for clave, valor in {"filas_malas": int(filas_malas), "tasa_rechazo": float(tasa_rechazo)}.items():
    try:
        dbutils.jobs.taskValues.set(key=clave, value=valor)
    except Exception as e:
        print(f"Ejecución fuera de un job: no se publicó {clave} ({e})")

print(f"filas_malas = {filas_malas}")
print(f"tasa_rechazo = {tasa_rechazo}")
display(spark.table(f"{CATALOG}.{SCHEMA}.movielens_rechazos").limit(20))

## Cierre

- Aplicaste cuatro reglas de calidad como indicadores booleanos auditables.
- Separaste datos válidos en Silver y observaciones inválidas en rechazos.
- Verificaste cómo `inyectar_error` fuerza la rama negativa sin editar código.
- Publicaste `filas_malas` y `tasa_rechazo` para controlar el DAG.